
# World Narration: Profile & Evaluation of Schematic Outputs for Visual Novel Worldbuilding

3 Visual Novel World Complexities Are Tested

1. StaticOne: N = 5 Character Sprites, 3 Background Images, 5 Character Dialogues - Static
2. StaticTwo: N = 15 Character Sprites, 5 Background Images, 15 Character Dialogues - Static
3. DynamicOne N = 5 Character Sprites, 3 Background Images, 5 Character Dialogues - Dynamic
4. DynamicTwo N = 15 Character Sprites, 5 Background Images, 15 Character Dialogues - Dynamic

Static - linear dialogues, single state tracking: character sprites, background imgs, character sprite static expression, zero branching choices
Dynamic - multi-turn dialogues, multi state tracking: character sprites, background imgs, character sprite dynamic expression, with branching choices

Each categories have worlds generated by an LLM that fills up schematic outputs.

Abstract:

Visual novels are often combined with text and images. Each components in
a visual novel serves as parts in a multimodal reasoning. Connecting text and images
could introduce concerns in inference. In the case of visual novels, balancing the entire
"world" during inference takes up a lot of resources, especially with how rich in storytelling and narration visual novels are. This paper aims to explore underlying resource expenditures during its inference and provide insights in how multimodal LLMs world building exhaust compute in the context of visual novels.

In [ ]:
# CONFIG
import psutil
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f"GPU Count: {torch.cuda.device_count()}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

CONFIG = {"SPRITES": [5, 15], "BG": [3, 5], "DIAG": ["STATIC", "DYNAMIC"]} 

system_benchmark_record = {
    "metadata": {
        "model_name": "model-name",
        "precision_format": "4-bit (bitsandbytes)" or "16-bit (BF16)",  
        "context_length_tokens": 0,
        "hooks_active": False
    },
    "system_metrics": {
        # Tracks physical RAM currently held by the Python process
        "process_rss_mb": 0.0,
        # Tracks virtual memory allocated to the process
        "process_vms_mb": 0.0,
        # Tracks total system RAM utilization percentage
        "system_ram_percent": 0.0,
        # Total swap space usage 
        "system_swap_used_mb": 0.0,
        # Time the CPU spent executing operating system/kernel-level instructions
        "cpu_kernel_time_sec": 0.0,
        # Time the CPU spent executing the actual Python code
        "cpu_user_time_sec": 0.0,
        # Number of times the OS thread scheduler paused the process for another task
        "voluntary_context_switches": 0,
        # Number of times the OS forced the process to pause 
        "involuntary_context_switches": 0,
        # Accumulated disk writes 
        "disk_write_bytes": 0
    },
    "hardware_accelerator": {
        # PyTorch-specific tracking since psutil doesn't natively read GPU VRAM
        "cuda_vram_allocated_mb": 0.0,
        "cuda_max_vram_reserved_mb": 0.0
    }
}

ideal_system_baselines = {
    "system_ram_percent": {
        "GOOD": 75.0,
        "WARNING": 89.0,
        "CRITICAL": 100.0
    },
    "system_swap_used_mb": {
        "GOOD": 0,
        "WARNING": 1023,
        "CRITICAL": 1024
    },
    "involuntary_context_switches": {
        "GOOD": 100,
        "WARNING": 1000,
        "CRITICAL": 5000
    },
    "disk_write_bytes_per_sec": {
        "GOOD": 0,
        "WARNING": 10485760,
        "CRITICAL": 104857600
    },
    "cuda_vram_utilization_percent": {
        "GOOD": 85.0,
        "WARNING": 94.0,
        "CRITICAL": 100.0
    }
}

def get_sys_metrics():
    process = psutil.Process()
    return {
        "rss_mb": process.memory_info().rss / (1024 * 1024),         # Physical memory
        "vms_mb": process.memory_info().vms / (1024 * 1024),         # Virtual memory
        "cpu_sys_time": psutil.cpu_times().system,                   # Time spent in kernel
        "ctx_switches": process.num_ctx_switches().voluntary,        # Thread context switches
        "disk_write_mb": psutil.disk_io_counters().write_bytes / (1024 * 1024)
    }


def compare_sys_metrics(sys_metric_name: str, sys_metric: int or float): 

    system_metrics_comparison = {
        "metadata": {
            "model_name": "model-name",
            "precision_format": "4-bit (bitsandbytes)" or "16-bit (BF16)",  
            "context_length_tokens": 0,
            "hooks_active": False
        },
        "comparison": {
            "system_ram_percent": {
                "GOOD": False,
                "WARNING": False,
                "CRITICAL": False
            },
            "system_swap_used_mb": {
                "GOOD": False,
                "WARNING": False,
                "CRITICAL": False
            },
            "involuntary_context_switches": {
                "GOOD": False,
                "WARNING": False,
                "CRITICAL": False
            },
            "disk_write_bytes_per_sec": {
                "GOOD": False,
                "WARNING": False,
                "CRITICAL": False
            },
            "cuda_vram_utilization_percent": {
                "GOOD": False,
                "WARNING": False,
                "CRITICAL": False
            }  
        }
    }
    
    for key, val in ideal_system_baseline[sys_metric_name].items():
        if val > sys_metrics:
             system_metrics_comparison['comparison'][sys_metric_name][key] = True
            break
        if val < sys_metrics:
            system_metrics_comparison['comparison'][sys_metric_name][key] = True
            break
        elif val == sys_metrics:
            system_metrics_comparison['comparison'][sys_metric_name][key] = True
            break

In [ ]:
    schema = {
        "type": "object",
        "additionalProperties": False,
        "required": ["background", "general", "avatar", "position", "world", "x", "y"],
        "properties": {
            "background": {
                "type": "string",
                "description": "Visual novel background. Asset ID or filename of the background image (e.g., 'bg_classroom_evening'). Do not output image data."
            },
            "general": {"$ref": "#/$defs/generalState"},
            "avatar": {"$ref": "#/$defs/avatarState"},
            "position": {"$ref": "#/$defs/positionState"},
            "world": {"$ref": "#/$defs/worldState"},
            "x": {
                "type": "number",
                "minimum": 0,
                "maximum": 1,
                "description": "Context importance coordinate. Must be a plain decimal number between 0.00 and 1.00 (e.g., 0.55). Do not use scientific notation, NaN, or text. X coordinate -> memoryspace"
            },
            "y": {
                "type": "number",
                "minimum": 0,
                "maximum": 1,
                "description": "Context relevance coordinate. Must be a plain decimal number between 0.00 and 1.00 (e.g., 0.55). Do not use scientific notation, NaN, or text. Y coordinate -> memoryspace"
            }
        },
        "$defs": {
            "generalState": {
                "type": "object",
                "additionalProperties": False,
                "required": ["id", "StateLinker", "StateOne", "StateTwo", "StateThree"],
                "description": "General narrative state for the current scene.",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "Unique ID for this general state (e.g., 'general_012')."
                    },
                    "StateLinker": {
                        "type": ["string", "null"],
                        "description": "ID of the previous generalState this one continues from, or null if this is the first state."
                    },
                    "StateOne": {"type": "string", "description": "First general state value."},
                    "StateTwo": {"type": "string", "description": "Second general state value."},
                    "StateThree": {"type": "string", "description": "Third general state value."}
                }
            },
            "avatarState": {
                "type": "object",
                "additionalProperties": False,
                "required": ["id", "StateLinker", "Expression", "Sprite"],
                "description": "Current state of the on-screen character.",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "Unique ID for this avatar state (e.g., 'avatar_012')."
                    },
                    "StateLinker": {
                        "type": ["string", "null"],
                        "description": "ID of the previous avatarState this one continues from, or null if this is the first state."
                    },
                    "Expression": {
                        "type": "string",
                        "description": "Character's facial expression (e.g., 'smiling', 'surprised', 'angry')."
                    },
                    "Sprite": {"$ref": "#/$defs/avatarSprite"}
                }
            },
            "avatarSprite": {
                "type": "object",
                "additionalProperties": False,
                "required": ["width", "height", "pixels", "z_buffer", "channel"],
                "description": "Sprite image metadata for the avatar.",
                "properties": {
                    "width": {"type": "integer", "minimum": 1, "description": "Sprite width in pixels."},
                    "height": {"type": "integer", "minimum": 1, "description": "Sprite height in pixels."},
                    "pixels": {
                        "type": "string",
                        "description": "Asset ID or path of the sprite image (e.g., 'sprites/alice_smile.png'). Do not output raw pixel data."
                    },
                    "z_buffer": {
                        "type": ["number", "null"],
                        "description": "Optional draw-order depth. Higher draws in front. Use null if not needed."
                    },
                    "channel": {
                        "type": ["integer", "null"],
                        "description": "Optional number of color channels (3 = RGB, 4 = RGBA). Use null if not needed."
                    }
                }
            },
            "positionState": {
                "type": "object",
                "additionalProperties": False,
                "required": ["id", "StateLinker", "x", "y", "z"],
                "description": "On-screen position of the avatar.",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "Unique ID for this position state (e.g., 'position_012')."
                    },
                    "StateLinker": {
                        "type": ["string", "null"],
                        "description": "ID of the previous positionState this one continues from, or null if this is the first state."
                    },
                    "x": {"type": "number", "description": "Horizontal position."},
                    "y": {"type": "number", "description": "Vertical position."},
                    "z": {"type": "number", "description": "Depth / layer position."}
                }
            },
            "worldState": {
                "type": "object",
                "additionalProperties": False,
                "required": ["id", "StateLinker", "Event", "Variables"],
                "description": "Global world state for the story.",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "Unique ID for this world state (e.g., 'world_012')."
                    },
                    "StateLinker": {
                        "type": ["string", "null"],
                        "description": "ID of the previous worldState this one continues from, or null if this is the first state."
                    },
                    "Event": {
                        "type": "string",
                        "description": "The current story event (e.g., 'festival_night_begins')."
                    },
                    "Variables": {
                        "type": "array",
                        "description": "World variables as key/value pairs (e.g., [{'key': 'weather', 'value': 'rain'}]).",
                        "items": {
                            "type": "object",
                            "additionalProperties": False,
                            "required": ["key", "value"],
                            "properties": {
                                "key": {"type": "string"},
                                "value": {"type": "string"}
                            }
                        }
                    }
                }
            }
        }
    }